In [ ]:
import os, random, numpy as np, torch
import torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


In [ ]:
DATASET_ROOT=r'D:\Processed_xView2_Dataset'
BATCH_SIZE=32
EPOCHS=15
LR=1e-4
NUM_WORKERS=2
random.seed(42); np.random.seed(42); torch.manual_seed(42)


In [ ]:
train_tf=transforms.Compose([
transforms.ToTensor(),
transforms.RandomHorizontalFlip(),
transforms.ColorJitter(brightness=0.2,contrast=0.2),
transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
val_tf=transforms.Compose([
transforms.ToTensor(),
transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

train_dataset=datasets.ImageFolder(os.path.join(DATASET_ROOT,'train'),transform=train_tf)
val_dataset=datasets.ImageFolder(os.path.join(DATASET_ROOT,'val'),transform=val_tf)
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
print(train_dataset.classes)


In [ ]:
labels=[y for _,y in train_dataset.samples]
w=compute_class_weight(class_weight='balanced',classes=np.unique(labels),y=labels)
w=torch.tensor(w,dtype=torch.float32).to(device)
model=models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1]=nn.Linear(model.classifier[1].in_features,4)
model=model.to(device)
criterion=nn.CrossEntropyLoss(weight=w)
optimizer=optim.Adam(model.parameters(),lr=LR)
scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',patience=2)


In [ ]:
train_loss_hist=[];val_loss_hist=[];train_acc_hist=[];val_acc_hist=[];best_acc=0
for epoch in range(EPOCHS):
    model.train(); tl=0;tc=0;tt=0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        out=model(x)
        loss=criterion(out,y)
        loss.backward(); optimizer.step()
        tl+=loss.item()*x.size(0)
        tc+=(out.argmax(1)==y).sum().item(); tt+=y.size(0)
    train_loss_hist.append(tl/tt); train_acc_hist.append(tc/tt)
    model.eval(); vl=0;vc=0;vt=0;all_preds=[];all_labels=[]
    with torch.no_grad():
        for x,y in val_loader:
            x,y=x.to(device),y.to(device)
            out=model(x); loss=criterion(out,y)
            vl+=loss.item()*x.size(0)
            p=out.argmax(1)
            vc+=(p==y).sum().item(); vt+=y.size(0)
            all_preds.extend(p.cpu().numpy()); all_labels.extend(y.cpu().numpy())
    val_loss_hist.append(vl/vt); val_acc_hist.append(vc/vt)
    scheduler.step(val_acc_hist[-1])
    print(epoch+1,train_acc_hist[-1],val_acc_hist[-1])
    if val_acc_hist[-1]>best_acc:
        best_acc=val_acc_hist[-1]
        torch.save(model.state_dict(),'best_model.pth')
print('Best',best_acc)


In [ ]:
plt.plot(train_acc_hist,label='Train');plt.plot(val_acc_hist,label='Val');plt.legend();plt.show()
plt.plot(train_loss_hist,label='Train');plt.plot(val_loss_hist,label='Val');plt.legend();plt.show()
print(classification_report(all_labels,all_preds,target_names=train_dataset.classes))
print(confusion_matrix(all_labels,all_preds))
